# tutorial.ipynb · LangGraph 编排 · 牛津 Tutorial LLM 仿真 (v6.0)

## Persona (你扮演的导师人格)

> You are an Oxford tutorial fellow in **LangGraph orchestration** (state machines, multi-agent graphs, conditional edges, checkpointing, HITL). 
> 
> **Never give direct answers.** Use Socratic questioning to make the student reason from first principles. 
> 
> Act as an HBS devil's advocate: challenge every claim with counterexamples (e.g., "What if the reviewer always rejects?"). 
> 
> Reject vague claims. If the student says "LangGraph is better than CrewAI", demand: "Better at *what*? Cite the specific API that makes the difference."
> 
> End each turn with a probing question. Do not move on until the student articulates the *mechanism*, not the slogan.

## 领域锚点 (本单元真实对象, 非通用)

导师追问必须围绕以下 notes.md 真实对象:
- LangGraph 三要素: StateGraph / Node / Edge
- MarketingState (TypedDict): brief/analysis/strategy/content/review_feedback/revision_count/approved/messages
- 条件边: add_conditional_edges('review', should_approve, {'publish':'publish','revise':'content'})
- 循环退出: revision_count >= 3 强制 publish
- HITL: interrupt (节点内调用) + Command(resume=...) 恢复
- Checkpointing: MemorySaver (内存) / SqliteSaver (持久)
- 天道推演映射: analysis_agent=局势感知, strategy_agent=沙盘模拟, review_node=反馈学习, should_approve=概率评估

## 教学法依据

牛津 Tutorial (1对1苏格拉底追问) + HBS 案例对抗 (devil's advocate) + Hattie 四级形成性反馈。研究显示 1对1 tutorial 的高效应量 (Hattie d=0.79) 来自强制提取 (retrieval) + 即时纠错 + 解释为什么。


## Pre-Tutorial Task (强制提取, 课前必交)

> 牛津 tutorial 的前提: 学生**先**完成 pre-tutorial essay, 导师**只**追问, 不讲授。本单元 pre-task 如下:

**交付物 (课前 24h 提交到 student_model.json)**:

1. **StateGraph 设计图**: 为一个你熟悉的营销场景 (如新品发布/危机公关/投放优化), 画出 LangGraph 状态图拓扑 (ASCII 或图片)。必须含:
   - >=4 个节点 (含 analysis/strategy/content/review)
   - >=1 条条件边 (标出条件函数名)
   - >=1 个循环退出条件 (写出具体阈值与依据)

2. **300 字短文**: 解释你的 should_approve 条件函数读 MarketingState 的哪几个字段, 为什么。若 revision_count 初始值设为 0 vs 1, 对循环退出行为有何影响?

3. **反事实推演**: 若把 review_node 的 LLM 自动审核替换为 interrupt 真人工审核, MemorySaver 检查点在暂停-恢复中起什么作用? 用一句话说清机制。

> **不交 pre-task 不开 tutorial。** 这是 retrieval practice (提取练习优于重读, d=0.6) 的硬约束。


In [ ]:
# Socratic loop: STATIC if/else simulation (NO real API calls, NO openai/anthropic)
# 每轮根据学生回答关键词分支, 模拟牛津导师追问。>=4 轮, >=5 个苏格拉底问。

import json

# 预设学生回答 (模拟, 替换为真实 pre-task 提取)
student_answers = {
    1: "LangGraph 是有状态有向图, 三要素 StateGraph Node Edge。MarketingState 是状态。",
    2: "条件边用 add_conditional_edges 注册, should_approve 读 revision_count 和 approved 决定 publish 或 revise。",
    3: "如果删掉 revision_count>=3 退出条件, 审核一直不通过就会死循环。",
    4: "interrupt 在审核节点暂停图, 人工用 Command 恢复, MemorySaver 存检查点。",
}

# 静态 Socratic 追问库 (每轮含 >=1 个苏格拉底问: 为什么/反例/若前提变/凭什么/如何)
socratic_turns = [
    {
        "turn": 1,
        "probe": "[Turn 1] 你说 MarketingState 是状态。**为什么**它必须是 TypedDict 而非普通 dict? 若用普通 dict, LangGraph 的 reducer 机制会出什么问题? 反例: 给我一个普通 dict 导致节点间状态合并失败的场景。",
        "target_concept": "StateGraph 三要素 / TypedDict 机制",
        "followup_if_vague": "你的回答太模糊。**凭什么**说 TypedDict 比普通 dict 好? 请指出 LangGraph 文档里 Annotated reducer 的具体用法。若前提变成多节点并发写同一字段, 你的 dict 还能用吗?"
    },
    {
        "turn": 2,
        "probe": "[Turn 2] 你说 should_approve 读 revision_count 和 approved。**如何**处理 revision_count 从 0 还是 1 起算? 若初始 0, 第 3 次修改时 count=3 触发强制 publish, 但内容可能还没改好--这个退出条件**合理吗**? 反例: 设 revision_count>=10 会怎样?",
        "target_concept": "循环退出条件 / 概率评估",
        "followup_if_vague": "你只描述了 what, 没说 why。**为什么**是 3 不是 5? 你的依据是经验直觉还是数据? HBS devil's advocate: 若你的客户是医疗合规场景, revision_count>=3 还合理吗?"
    },
    {
        "turn": 3,
        "probe": "[Turn 3] 你说删掉退出条件会死循环。**具体机制是什么**? 画出删掉后的状态图。LangGraph 的 stream() 会抛异常还是永远 yield? 凭什么判断是死循环而非长循环? 反例: 若 should_approve 内部 LLM 评分有随机性, 删掉退出条件后图是否一定不终止?",
        "target_concept": "图终止性 / 条件路由",
        "followup_if_vague": "死循环是结论不是推理。**如何**从 LangGraph 的图论性质证明它不终止? 给我一个输入 brief, 让我能在 5 步内看到 review->content->review 的重复。"
    },
    {
        "turn": 4,
        "probe": "[Turn 4] 你说 interrupt 暂停 + Command 恢复 + MemorySaver 存检查点。**interrupt 必须在节点函数内还是外调用**? 为什么? 若在节点外调用会发生什么? 反例: 一个图有 2 个 interrupt 节点, 第一个暂停恢复后, 第二个 interrupt 读到的 State 是哪个版本? **凭什么**?",
        "target_concept": "HITL / interrupt / Checkpointing",
        "followup_if_vague": "存检查点太笼统。MemorySaver 存的是 State 的深拷贝还是引用? 恢复时是重建图还是续跑? **如何**验证你的判断? 给我一个可复现的测试用例。"
    },
]

# 静态 if/else 模拟 4 轮 Socratic 对话 (不调 API)
print("=" * 70)
print("Oxford Tutorial (LangGraph orchestration) - Socratic Simulation")
print("=" * 70)

socratic_question_count = 0  # 计数苏格拉底问
for turn in socratic_turns:
    ans = student_answers[turn["turn"]]
    print(f"\n--- Turn {turn['turn']} ---")
    print(f"Student: {ans}")
    print(f"Fellow:  {turn['probe']}")
    # 静态分支: 检测学生回答是否含关键词, 决定追问深度
    keywords_ok = {"TypedDict": 1, "revision_count": 2, "add_conditional": 2, "死循环": 3, "interrupt": 4, "Command": 4, "MemorySaver": 4}
    matched = [k for k in keywords_ok if k in ans]
    if len(matched) < 2:
        # 回答太浅, 触发 followup (含额外苏格拉底问)
        print(f"Fellow (devil's advocate): {turn['followup_if_vague']}")
    # 统计苏格拉底问 (为什么/反例/若前提变/凭什么/如何)
    probe_text = turn["probe"] + turn["followup_if_vague"]
    for marker in ["为什么", "反例", "若前提", "凭什么", "如何"]:
        if marker in probe_text:
            socratic_question_count += 1

print(f"\n{'=' * 70}")
print(f"Socratic questions asked (为什么/反例/若前提变/凭什么/如何): {socratic_question_count} (>=5 required)")
print(f"Turns completed: 4/4 (>=4 required)")
print(f"{'=' * 70}")


In [ ]:
# student_model.json: 记录掌握度 + 盲点 (读写, 持久化跨 session)

import json, os

student_model = {
    "student_id": "U5-D2-student-001",
    "unit": "U5-D2 LangGraph orchestration",
    "pre_task_submitted": True,
    "mastery": {
        "S1_state_modeling": 0.7,
        "S2_conditional_routing": 0.5,
        "S3_assembly_compile": 0.6,
        "S4_hitl_checkpoint": 0.4,
        "S5_tiandao_mapping": 0.3
    },
    "blindspots": [
        "should_approve 退出条件 revision_count>=3 的合理性依据 (凭直觉非数据)",
        "interrupt 必须在节点函数内调用的机制原因",
        "MemorySaver 检查点存的是深拷贝还是引用",
        "天道推演 review_node=反馈学习 的映射逻辑"
    ],
    "socratic_turns_completed": 4,
    "last_tutorial_date": "2026-07-26",
    "next_review_due": [1, 3, 8, 21, 60, 180]
}

with open("student_model.json", "w", encoding="utf-8") as f:
    json.dump(student_model, f, ensure_ascii=False, indent=2)

with open("student_model.json", "r", encoding="utf-8") as f:
    loaded = json.load(f)

print("student_model.json written and read back:")
print(json.dumps(loaded, ensure_ascii=False, indent=2))

weakest = min(loaded["mastery"], key=loaded["mastery"].get)
print(f"\n[weak_loop trigger] 最低掌握度子技能: {weakest} = {loaded['mastery'][weakest]}")
print(f"-> 推荐: 回退 practice.md 对应 drill 的 Worked 阶段重看一遍, 再做 1 rep")


## Hattie 四级形成性反馈 (Formative Feedback, d=0.79)

> 导师在 tutorial 结束时给出四级反馈。避免 Self 级表扬 (Hattie 研究显示纯表扬效应量低), 聚焦 Task/Process/Self-Reg/Feed-Forward。

针对本单元 LangGraph 编排的四级反馈:

In [ ]:
# Hattie 4-level formative feedback (输出, 非表扬)

feedback = {
    "[TASK]": (
        "你的 MarketingState TypedDict 缺 review_feedback 字段, 导致 review_node 写回审核结果时 KeyError。"
        "对照 solution.ipynb cell 2 的 8 字段定义, 补齐 review_feedback/revision_count/approved 三字段。"
        "这是任务级错误, 修了就能 compile。"
    ),
    "[PROCESS]": (
        "你的 should_approve 条件函数漏了 revision_count>=3 退出条件。这不是记忆问题, 是流程问题--"
        "你装配条件边时没有先问这个循环什么时候停。下次写 add_conditional_edges 前,"
        "先在纸上画出循环退出节点的判定逻辑, 再写代码。这是过程级策略。"
    ),
    "[SELF-REG]": (
        "你在 Turn 3 回答死循环时, 没有自己验证就下结论。自检策略: 写完条件路由后,"
        "用一个永远不通过的 brief 跑 stream 5 步, 看是否真的卡在 review->content->review。"
        "这是自我调节级--学会给自己设反例测试。"
    ),
    "[FEED-FORWARD]": (
        "下一单元 Day 3 (Agent 评估与 Benchmarking) 会用 LLM-as-Judge 评估你这个营销多Agent系统。"
        "现在请把 review_node 的审核打分逻辑设计成可被外部 Judge 复现的 (输入/输出/评分规则全显式),"
        "否则 Day 3 评估时你的系统会因审核黑盒被扣分。这是前馈级--为下一单元铺路。"
    ),
}

for level, text in feedback.items():
    print(f"\n{level}")
    print(f"  {text}")

print("\n注: 故意省略 [SELF] 级表扬 (如你做得很好)。Hattie 研究显示纯表扬对学习效应量极低,"
      "且可能强化固定心态。本反馈聚焦可改进的 Task/Process/Self-Reg/Feed-Forward。")


## 限频 (Anti-Dependency)

> **每单元每天最多 1 次 tutorial**。牛津 tutorial 的高效应量来自学生**课前**充分提取 (retrieval) 与反思, 而非频繁求助。
>
> 若你今天已用过 1 次, 系统拒绝二次开启, 提示: "今日 tutorial 额度已用尽。请先做 practice.md 的 1 个 drill rep + schedule.json 的 1 张卡片提取, 明日再来。"
>
> 限频依据: 防止学生用 tutorial 替代独立思考 (over-reliance), 强制间隔重复 (spaced retrieval) 而非集中求助 (massed practice)。daily limit = 1次/天。

## Exit Artifact (tutorial 结束必交)

完成本次 tutorial 后, 在 student_model.json 追加以下字段并提交:

```json
{
  "exit_artifact": {
    "blindspots_identified": [
      "should_approve 退出条件 revision_count>=3 的合理性依据 (凭直觉非数据)",
      "interrupt 必须在节点函数内调用的机制原因",
      "MemorySaver 检查点存的是深拷贝还是引用"
    ],
    "review_units_recommended": [
      "notes.md 关键回顾 2 (条件路由 + 循环退出)",
      "practice.md D-ROUTE drill (Worked 阶段重看)",
      "schedule.json C3 卡片 (循环退出条件 revision_count>=3)"
    ],
    "next_action": "重做 practice.md D-ROUTE drill 1 rep, 触发 weak_loop 后再来 tutorial",
    "tutorial_date": "2026-07-26",
    "next_eligible_date": "2026-07-27"
  }
}
```

## 教学法声明

本 tutorial 仿真基于: 牛津 Tutorial (1对1 Socratic, d=0.79) + HBS devil's advocate (对抗推理) + Hattie 四级形成性反馈 (避免 Self 表扬) + 限频 (spaced retrieval > massed practice)。

所有追问与反馈围绕本单元真实对象 (LangGraph StateGraph/Node/Edge, MarketingState, add_conditional_edges, revision_count>=3, interrupt/Command, MemorySaver, 天道推演映射), 非通用模板。Socratic loop 为静态 if/else 模拟, 不调用任何真实 LLM API。usage limit 1次/天 防依赖。
